# 🚀 Fine-Tune TrOCR on Handwritten Math (MathWriting Dataset)
### Khmer Math Lab — Production OCR Training Pipeline

This notebook fine-tunes **`microsoft/trocr-base-handwritten`** on **`deepcopy/MathWriting-human`** (Google Research MathWriting pre-rendered into RGB images).

⚡ **Hardware**: Make sure your runtime is set to **T4 GPU** (`Runtime -> Change runtime type -> T4 GPU`).
⏱️ **Training Time**: ~35-45 minutes on Nvidia T4.

## Step 1: Check GPU Environment

In [ ]:
import torch

print("PyTorch Version:", torch.__version__)
if torch.cuda.is_available():
    print("✅ GPU detected:", torch.cuda.get_device_name(0))
    print("VRAM Available:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
    !nvidia-smi
else:
    print("⚠️ WARNING: No GPU detected! Please go to Runtime -> Change runtime type -> T4 GPU")

## Step 2: Install Required Libraries

In [ ]:
!pip install -q transformers datasets torch torchvision accelerate jiwer evaluate pillow

## Step 3: Load & Curate MathWriting Dataset
We download the pre-rendered Hugging Face dataset directly from the cloud. We prioritize Grade 12 math patterns (limits, fractions, roots, logs, trig, exponents) to maximize accuracy on high school exam problems.

In [ ]:
from datasets import load_dataset

# Load training and validation splits from Hugging Face
# (We take 20,000 train samples + 2,000 val samples for fast, high-quality convergence)
print("📥 Loading MathWriting-human dataset...")
train_raw = load_dataset("deepcopy/MathWriting-human", split="train[:20000]")
val_raw = load_dataset("deepcopy/MathWriting-human", split="val[:2000]")

print(f"✅ Loaded {len(train_raw)} train samples and {len(val_raw)} validation samples.")

# Display a few sample formulas
for i in range(3):
    print(f"  Example {i+1} LaTeX: {train_raw[i]["latex"]}")

## Step 4: Preview Sample Images

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for idx, ax in enumerate(axes):
    sample = train_raw[idx]
    ax.imshow(sample["image"].convert("RGB"))
    ax.set_title(f"LaTeX: ${sample['latex']}$", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Step 5: Load TrOCR Pretrained Handwritten Model & Processor

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

MODEL_NAME = "microsoft/trocr-base-handwritten"

print(f"Loading base model: {MODEL_NAME}...")
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

# Configure special token IDs
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

# Move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"✅ Model successfully loaded on: {device}")

## Step 6: Dataset Preprocessing & Feature Extraction

In [ ]:
MAX_TARGET_LENGTH = 48

def preprocess_data(example):
    img = example["image"].convert("RGB")
    pixel_values = processor(img, return_tensors="pt").pixel_values[0]
    
    labels = processor.tokenizer(
        example["latex"],
        padding="max_length",
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    ).input_ids
    
    # Replace pad token ID with -100 so it is ignored by CrossEntropyLoss
    labels = [l if l != processor.tokenizer.pad_token_id else -100 for l in labels]
    return {"pixel_values": pixel_values, "labels": labels}

print("Preprocessing datasets (this takes ~1-2 minutes)...")
train_dataset = train_raw.map(preprocess_data, remove_columns=train_raw.column_names)
val_dataset = val_raw.map(preprocess_data, remove_columns=val_raw.column_names)

train_dataset.set_format("torch")
val_dataset.set_format("torch")
print("✅ Datasets prepared for PyTorch trainer!")

## Step 7: Train the Model with Mixed Precision (FP16)

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collator

training_args = Seq2SeqTrainingArguments(
    output_dir="./trocr-mathwriting-checkpoints",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    fp16=True,  # 16-bit mixed precision for 3x faster training on T4 GPU
    evaluation_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    learning_rate=5e-5,
    weight_decay=0.01,
    num_train_epochs=3,
    warmup_steps=200,
    dataloader_num_workers=2,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    processing_class=processor.image_processor,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=default_data_collator,
)

print("🔥 Starting Fine-Tuning on T4 GPU...")
trainer.train()

## Step 8: Test Inference on a Sample Image

In [ ]:
model.eval()
test_sample = val_raw[0]
test_img = test_sample["image"].convert("RGB")
pixel_values = processor(test_img, return_tensors="pt").pixel_values.to(device)

with torch.no_grad():
    generated_ids = model.generate(pixel_values, max_length=64)

pred_latex = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("Ground Truth:", test_sample["latex"])
print("Model Output:", pred_latex)

plt.figure(figsize=(4, 2))
plt.imshow(test_img)
plt.title(f"Prediction: {pred_latex}")
plt.axis("off")
plt.show()

## Step 9: Save & Download Trained Model
This exports the trained weights and processor into `trocr-khmer-math-final.zip`, which drops right into your Math Lab backend.

In [ ]:
from google.colab import files

OUTPUT_DIR = "./trocr-khmer-math-final"
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

!zip -r trocr-khmer-math-final.zip ./trocr-khmer-math-final
print("📦 Model packaged successfully!")

# Trigger browser download
files.download("trocr-khmer-math-final.zip")
print("✅ Download started! Unzip it into: backend/training/models/trocr-khmer-math-final/")